# 배치 단위 제어전략 통계검정

`merged_data_ko.csv`에서 배치별 성과지표를 만든 뒤 RC·OC·APC의 생산 성과 차이를 검정한다. 0.2시간 간격의 행을 독립 표본으로 취급하지 않고 **100개 배치**를 독립 분석 단위로 사용한다.

## tl;dr

- 최종농도 평균: RC 24.32, OC 21.51, APC 29.41 g/L. Welch ANOVA Holm 보정 p=0.000014.
- 총수확량 평균: RC 2,912,750, OC 2,833,113, APC 3,484,283. Welch ANOVA Holm 보정 p=0.000014.
- Games–Howell 결과 APC는 최종농도와 총수확량 모두 RC·OC보다 유의하게 높았고, RC–OC 차이는 유의하지 않았다.
- 종료시간 평균은 RC 228.83, OC 229.23, APC 224.67시간이며 전략 차이 근거가 없었다(p=0.474937).
- 이는 관찰된 연관성이며 인과관계로 해석하지 않는다.

이 노트북은 CSV를 생성하거나 원본 값을 변경하지 않는다.

In [6]:
import sys

print(sys.executable)

d:\QAQC 본캠프\실전 프로젝트\Code\asdf\.venv.3.12\Scripts\python.exe


In [7]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "statsmodels"
])

0

## 1. 라이브러리 불러오기

In [8]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway

ALPHA = 0.05
STRATEGY_ORDER = ['RC', 'OC', 'APC']

## 2. 파일 불러오기

분석 파일만 읽고 원본 DataFrame은 수정하지 않는다.

In [9]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        target = candidate / 'data' / 'interim' / 'merged_data_ko.csv'
        if target.exists():
            return candidate
    raise FileNotFoundError('data/interim/merged_data_ko.csv를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / 'data' / 'interim' / 'merged_data_ko.csv'

merged_data_ko = pd.read_csv(DATA_PATH)
original_shape = merged_data_ko.shape

print(f'파일: {DATA_PATH}')
print(f'크기: {merged_data_ko.shape[0]:,}행 × {merged_data_ko.shape[1]}열')
print(f'배치 수: {merged_data_ko["배치번호"].nunique()}개')

파일: d:\QAQC 본캠프\실전 프로젝트\Code\asdf\penicillin-process-analysis\data\interim\merged_data_ko.csv
크기: 113,935행 × 39열
배치 수: 100개


## 3. 배치별 성과지표 만들기

배치번호를 기준으로 RC(1~30), OC(31~60), APC(61~90), Fault(91~100)를 구분한다. OUR 음수는 클리핑하지 않고 원본값으로 음수 발생 비율만 계산한다.

성과지표:

- 최종·최대 페니실린 농도
- 농도 유지율 = 최종농도 / 최대농도
- 총수확량
- 생산성 = 최종농도 / 종료시간
- 농도 AUC
- 후기 20% 농도 기울기
- 공정 종료시간
- OUR 음수 발생 비율

In [15]:
def get_strategy(batch_number: int) -> str:
    if 1 <= batch_number <= 30:
        return 'RC'
    if 31 <= batch_number <= 60:
        return 'OC'
    if 61 <= batch_number <= 90:
        return 'APC'
    if 91 <= batch_number <= 100:
        return 'Fault'
    return 'Invalid'


def summarize_batch(batch_number: int, batch_data: pd.DataFrame) -> pd.Series:
    batch_data = batch_data.sort_values('발효시간(h)')
    time = batch_data['발효시간(h)'].to_numpy()
    concentration = batch_data['페니실린농도(g/L)'].to_numpy()

    end_time = float(time[-1])
    final_concentration = float(concentration[-1])
    max_concentration = float(np.max(concentration))
    retention = final_concentration / max_concentration if max_concentration != 0 else np.nan
    productivity = final_concentration / end_time if end_time != 0 else np.nan
    concentration_auc = float(np.trapezoid(concentration, time))

    progress = time / end_time
    late_mask = progress >= 0.8
    late_slope = float(np.polyfit(time[late_mask], concentration[late_mask], 1)[0])

    return pd.Series({
        '배치번호': int(batch_number),
        '전략': get_strategy(batch_number),
        '최종농도(g/L)': final_concentration,
        '최대농도(g/L)': max_concentration,
        '농도유지율': retention,
        '총수확량(kg)': float(batch_data['총수확량(kg)'].iloc[0]),
        '생산성(g/L/h)': productivity,
        '농도AUC': concentration_auc,
        '후기농도기울기': late_slope,
        '종료시간(h)': end_time,
        'OUR평균_raw': float(batch_data['산소소모율(g/min)'].mean()),
        'OUR음수비율': float(batch_data['산소소모율(g/min)'].lt(0).mean()),
    })


batch_metrics = pd.DataFrame([
    summarize_batch(batch_number, batch_data)
    for batch_number, batch_data in merged_data_ko.groupby('배치번호', sort=True)
])

strategy_counts = batch_metrics['전략'].value_counts().reindex(['RC', 'OC', 'APC', 'Fault'])
assert len(batch_metrics) == 100
assert strategy_counts.to_dict() == {'RC': 30, 'OC': 30, 'APC': 30, 'Fault': 10}

display(batch_metrics.head(100))
display(strategy_counts.rename('배치수').to_frame())

,배치번호,전략,최종농도(g/L),최대농도(g/L),농도유지율,총수확량(kg),생산성(g/L/h),농도AUC,후기농도기울기,종료시간(h),OUR평균_raw,OUR음수비율
0,1,RC,29.3730,29.409,0.998776,2786400.0,0.129969,3360.316134,0.074064,226.0,1.327662,0.007080
1,2,RC,30.4160,30.416,1.000000,2326000.0,0.132243,3790.272885,0.063440,230.0,1.375537,0.006957
2,3,RC,17.4280,28.103,0.620147,2675300.0,0.062691,4483.852396,-0.102661,278.0,1.291738,0.002158
3,4,RC,15.1070,23.740,0.636352,1886700.0,0.065683,2982.817961,-0.113900,230.0,1.045671,0.006087
4,5,RC,28.1720,28.172,1.000000,3562900.0,0.157385,2259.168647,0.128194,179.0,1.465969,0.008939
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,Fault,21.1040,27.760,0.760231,3595000.0,0.091757,3490.866465,-0.155936,230.0,1.200941,0.011304
96,97,Fault,16.9580,23.437,0.723557,3894300.0,0.075369,2737.714889,-0.115654,225.0,1.205496,0.008889
97,98,Fault,32.8020,32.809,0.999787,1730600.0,0.142617,3943.492056,0.066309,230.0,1.267638,0.015652
98,99,Fault,5.4669,13.709,0.398782,890830.0,0.021780,1915.010948,-0.048977,251.0,0.853677,0.005578


,배치수
전략,
RC,30
OC,30
APC,30
Fault,10


전략별 배치 수는 RC 30개, OC 30개, APC 30개, Fault 10개입니다. 

## 4. 기술통계

제어전략 성과 비교에서는 Fault 배치를 제외한다. 사전에 정한 주요 결과지표는 최종농도, 총수확량, 종료시간이다.

In [11]:
PRIMARY_METRICS = ['최종농도(g/L)', '총수확량(kg)', '종료시간(h)']
strategy_batches = batch_metrics.loc[batch_metrics['전략'].isin(STRATEGY_ORDER)].copy()
strategy_batches['전략'] = pd.Categorical(
    strategy_batches['전략'], categories=STRATEGY_ORDER, ordered=True
)

descriptive_statistics = (
    strategy_batches.groupby('전략', observed=True)[PRIMARY_METRICS]
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)
display(descriptive_statistics.round(4))

최종농도(g/L)                                           총수확량(kg)  \
        count     mean     std  median      min     max    count   
전략                                                                 
RC         30  24.3152  8.7016  27.994   6.6957  36.163       30   
OC         30  21.5084  8.0264  21.040   5.8802  34.756       30   
APC        30  29.4065  2.7265  29.889  21.5960  33.209       30   

                                                                종료시간(h)  \
             mean          std     median        min        max   count   
전략                                                                        
RC   2.912750e+06  767576.4447  2782050.0  1842400.0  4083100.0      30   
OC   2.833113e+06  679929.2406  2792650.0  1474100.0  4196000.0      30   
APC  3.484283e+06  307980.1080  3479350.0  2830800.0  4044300.0      30   

                                             
         mean      std median    min    max  
전략                                           
RC   228.8333  24.3000  230.0  179.0  290.0  
OC   229.2333  14.6068  230.0  193.0  262.0  
APC  224.6667  15.3518  230.0  167.0  241.0

관찰된 평균만 보면 APC가 가장 높고, OC가 가장 낮습니다.
RC와 OC는 표준편차가 각각 약 8.70, 8.03으로 큰 반면 APC는 약 2.73으로 작습니다. 즉 APC는 평균 성과가 높을 뿐 아니라 배치 간 결과도 상대적으로 안정적이었습니다.

APC가 RC보다 약 571,533, OC보다 약 651,170 높습니다.
다만 값이 수백만 kg으로 표시되므로 실제 단위가 kg인지, g인지, 시뮬레이션 스케일인지 확인해야 합니다. 단위 확인 전에는 실무적인 생산량으로 직접 해석하면 안 됩니다.

APC가 약 4시간 짧아 보이지만, 이 차이가 우연한 변동보다 크다고 판단할 통계적 근거는 없습니다.

## 5. Levene·Welch ANOVA·Kruskal–Wallis

Levene 검정은 분산 동질성을 진단한다. 분산이 달라도 사용할 수 있는 Welch ANOVA를 주 검정으로 사용하며, 비모수 강건성 확인을 위해 Kruskal–Wallis 결과도 함께 제시한다. 세 주요 지표의 Welch p값에는 Holm 보정을 추가한다.

In [12]:
omnibus_rows = []

for metric in PRIMARY_METRICS:
    samples = [
        strategy_batches.loc[strategy_batches['전략'].eq(strategy), metric].dropna().to_numpy()
        for strategy in STRATEGY_ORDER
    ]

    levene = stats.levene(*samples, center='median')
    welch = anova_oneway(samples, use_var='unequal')
    kruskal = stats.kruskal(*samples)
    total_n = sum(len(sample) for sample in samples)
    epsilon_squared = max(0.0, (kruskal.statistic - len(samples) + 1) / (total_n - len(samples)))

    omnibus_rows.append({
        '지표': metric,
        'Levene_W': levene.statistic,
        'Levene_p': levene.pvalue,
        'Welch_F': welch.statistic,
        'Welch_df1': welch.df_num,
        'Welch_df2': welch.df_denom,
        'Welch_p': welch.pvalue,
        'Kruskal_H': kruskal.statistic,
        'Kruskal_p': kruskal.pvalue,
        'Kruskal_epsilon2': epsilon_squared,
    })

omnibus_results = pd.DataFrame(omnibus_rows)
omnibus_results['Welch_p_Holm'] = multipletests(
    omnibus_results['Welch_p'], method='holm'
)[1]
omnibus_results['Welch_판정'] = np.where(
    omnibus_results['Welch_p_Holm'] < ALPHA, '전략 간 차이 있음', '차이 근거 부족'
)

display(omnibus_results.round(6))

,지표,Levene_W,Levene_p,Welch_F,Welch_df1,Welch_df2,Welch_p,Kruskal_H,Kruskal_p,Kruskal_epsilon2,Welch_p_Holm,Welch_판정
0,최종농도(g/L),12.259872,0.000020,16.112647,2.0,45.635462,0.000005,13.656068,0.001083,0.133978,0.000014,전략 간 차이 있음
1,총수확량(kg),10.981534,0.000056,15.965686,2.0,49.186990,0.000005,15.150867,0.000513,0.151159,0.000014,전략 간 차이 있음
2,종료시간(h),1.737239,0.182055,0.754552,2.0,56.055720,0.474937,0.593920,0.743074,0.000000,0.474937,차이 근거 부족


3. Levene 검정
Levene 검정은 전략별 분산이 같은지 확인합니다.
귀무가설: 세 전략의 분산이 같다.
p < 0.05: 분산이 같다고 보기 어렵다.
p ≥ 0.05: 분산 차이를 확인하지 못했다.
지표	Levene p값	판단
최종농도	0.000020	분산이 다름
총수확량	0.000056	분산이 다름
종료시간	0.182055	분산 차이 근거 없음


최종농도와 총수확량은 등분산 가정이 깨졌습니다. 따라서 일반 ANOVA 대신 Welch ANOVA를 사용한 현재 방법이 적절합니다.
4. Welch ANOVA
Welch ANOVA는 세 전략의 평균이 모두 같은지 검정합니다.
귀무가설: RC·OC·APC 평균이 모두 같다.
대립가설: 적어도 한 전략의 평균이 다르다.
지표	Welch F	Holm 보정 p값	판단
최종농도	16.113	0.000014	전략별 차이 있음
총수확량	15.966	0.000014	전략별 차이 있음
종료시간	0.755	0.474937	차이 근거 부족


따라서 다음과 같이 판단합니다.
최종농도: 귀무가설 기각
총수확량: 귀무가설 기각
종료시간: 귀무가설을 기각하지 못함
종료시간 결과는 “세 전략이 완전히 같다”는 뜻이 아닙니다. 현재 30개씩의 배치로는 유의한 차이를 확인하지 못했다는 뜻입니다.
Welch_p_Holm은 3개 결과지표를 동시에 검정하면서 우연한 유의성을 줄인 보정 p값입니다. 최종 판단에는 원래 p값보다 이 값을 사용하는 것이 안전합니다.
5. Kruskal–Wallis 검정
Kruskal–Wallis는 정규분포와 등분산에 덜 의존하는 비모수 검정입니다.
지표	H	p값	ε²
최종농도	13.656	0.001083	0.134
총수확량	15.151	0.000513	0.151
종료시간	0.594	0.743074	0.000


Welch ANOVA와 같은 방향입니다.
최종농도와 총수확량은 전략별 차이가 있음
종료시간은 차이 근거가 없음
ε²는 비모수 효과크기입니다. 최종농도 0.134는 중간~큰 수준, 총수확량 0.151은 큰 수준으로 볼 수 있습니다. 종료시간은 사실상 효과가 없습니다.

## 6. Games–Howell 사후검정

등분산을 가정하지 않는 Games–Howell로 전략 쌍을 비교한다. 평균차는 앞 전략에서 뒤 전략을 뺀 값이다. Hedges' g와 95% 신뢰구간을 함께 제시한다.

In [13]:
def hedges_g(sample_a: np.ndarray, sample_b: np.ndarray) -> float:
    n_a, n_b = len(sample_a), len(sample_b)
    pooled_variance = (
        ((n_a - 1) * np.var(sample_a, ddof=1) + (n_b - 1) * np.var(sample_b, ddof=1))
        / (n_a + n_b - 2)
    )
    if pooled_variance == 0:
        return np.nan
    cohen_d = (np.mean(sample_a) - np.mean(sample_b)) / np.sqrt(pooled_variance)
    correction = 1 - 3 / (4 * (n_a + n_b) - 9)
    return float(correction * cohen_d)


def games_howell(data: pd.DataFrame, metric: str, group_column: str = '전략') -> pd.DataFrame:
    rows = []
    group_count = data[group_column].nunique()

    for group_a, group_b in combinations(STRATEGY_ORDER, 2):
        sample_a = data.loc[data[group_column].eq(group_a), metric].dropna().to_numpy()
        sample_b = data.loc[data[group_column].eq(group_b), metric].dropna().to_numpy()
        n_a, n_b = len(sample_a), len(sample_b)
        mean_a, mean_b = np.mean(sample_a), np.mean(sample_b)
        var_a, var_b = np.var(sample_a, ddof=1), np.var(sample_b, ddof=1)
        mean_difference = mean_a - mean_b
        standard_error = np.sqrt(var_a / n_a + var_b / n_b)
        df = standard_error**4 / (
            (var_a / n_a) ** 2 / (n_a - 1) + (var_b / n_b) ** 2 / (n_b - 1)
        )
        q_statistic = np.sqrt(2) * abs(mean_difference) / standard_error
        p_value = stats.studentized_range.sf(q_statistic, group_count, df)
        q_critical = stats.studentized_range.ppf(1 - ALPHA, group_count, df)
        margin = q_critical * standard_error / np.sqrt(2)

        rows.append({
            '지표': metric,
            '비교': f'{group_a} - {group_b}',
            '평균_A': mean_a,
            '평균_B': mean_b,
            '평균차(A-B)': mean_difference,
            '95%CI_하한': mean_difference - margin,
            '95%CI_상한': mean_difference + margin,
            'GamesHowell_p': p_value,
            'Hedges_g': hedges_g(sample_a, sample_b),
            '판정': '유의한 차이' if p_value < ALPHA else '차이 근거 부족',
        })

    return pd.DataFrame(rows)


posthoc_results = pd.concat(
    [games_howell(strategy_batches, metric) for metric in PRIMARY_METRICS],
    ignore_index=True,
)
display(posthoc_results.round(6))

,지표,비교,평균_A,평균_B,평균차(A-B),95%CI_하한,95%CI_상한,GamesHowell_p,Hedges_g,판정
0,최종농도(g/L),RC - OC,2.431524e+01,2.150840e+01,2.806840,-2.392738,8.006418,0.401725,0.330958,차이 근거 부족
1,최종농도(g/L),RC - APC,2.431524e+01,2.940653e+01,-5.091290,-9.167487,-1.015093,0.011613,-0.779347,유의한 차이
2,최종농도(g/L),OC - APC,2.150840e+01,2.940653e+01,-7.898130,-11.682864,-4.113396,0.000033,-1.300546,유의한 차이
3,총수확량(kg),RC - OC,2.912750e+06,2.833113e+06,79636.666667,-370844.346723,530117.680057,0.905247,0.108405,차이 근거 부족
4,총수확량(kg),RC - APC,2.912750e+06,3.484283e+06,-571533.333333,-939755.243964,-203311.422702,0.001505,-0.964591,유의한 차이
5,총수확량(kg),OC - APC,2.833113e+06,3.484283e+06,-651170.000000,-982729.527142,-319610.472858,0.000069,-1.217710,유의한 차이
6,종료시간(h),RC - OC,2.288333e+02,2.292333e+02,-0.400000,-12.922874,12.122874,0.996714,-0.019693,차이 근거 부족
7,종료시간(h),RC - APC,2.288333e+02,2.246667e+02,4.166667,-8.516993,16.850327,0.708460,0.202345,차이 근거 부족
8,종료시간(h),OC - APC,2.292333e+02,2.246667e+02,4.566667,-4.739686,13.873020,0.469562,0.300813,차이 근거 부족


6. Games–Howell 사후검정
Welch ANOVA는 “셋 중 하나가 다르다”까지만 알려줍니다. Games–Howell은 어떤 전략끼리 다른지 확인합니다.
평균차는 앞 전략 - 뒤 전략입니다. 음수면 뒤 전략이 더 높습니다.
최종농도
비교	평균차	95% 신뢰구간	p값	Hedges’ g	판단
RC−OC	+2.81	−2.39~8.01	0.401725	0.331	차이 근거 부족
RC−APC	−5.09	−9.17~−1.02	0.011613	−0.779	APC가 높음
OC−APC	−7.90	−11.68~−4.11	0.000033	−1.301	APC가 높음


판단:
APC는 RC보다 평균 약 5.09g/L 높음
APC는 OC보다 평균 약 7.90g/L 높음
RC와 OC는 평균 차이가 관찰됐지만 신뢰구간에 0이 포함돼 확정하기 어려움
Hedges’ g 절댓값 기준으로 APC−RC는 중간~큰 차이, APC−OC는 매우 큰 차이입니다.
총수확량
비교	평균차	95% 신뢰구간	p값	Hedges’ g	판단
RC−OC	+79,637	−370,844~530,118	0.905247	0.108	차이 근거 부족
RC−APC	−571,533	−939,755~−203,311	0.001505	−0.965	APC가 높음
OC−APC	−651,170	−982,730~−319,610	0.000069	−1.218	APC가 높음


판단:
APC의 총수확량은 RC·OC보다 통계적으로 높음
두 비교 모두 효과크기가 큼
RC와 OC의 차이는 매우 작고 불확실함
종료시간
모든 비교의 p값이 0.46 이상이고 신뢰구간이 0을 포함합니다.
RC−OC: −0.4시간
RC−APC: +4.17시간
OC−APC: +4.57시간
APC가 평균적으로 조금 짧게 운전됐지만 통계적으로 확정할 수 없습니다.

## 7. 결과 요약

In [14]:
mean_table = (
    strategy_batches.groupby('전략', observed=True)[PRIMARY_METRICS]
    .mean()
    .reindex(STRATEGY_ORDER)
)
significant_pairs = posthoc_results.loc[
    posthoc_results['GamesHowell_p'] < ALPHA,
    ['지표', '비교', '평균차(A-B)', '95%CI_하한', '95%CI_상한', 'GamesHowell_p', 'Hedges_g'],
]

print('전략별 평균')
display(mean_table.round(4))
print('Holm 보정 후 Welch ANOVA')
display(omnibus_results[['지표', 'Welch_F', 'Welch_p', 'Welch_p_Holm', 'Welch_판정']].round(6))
print('유의한 Games–Howell 비교')
display(significant_pairs.round(6))

assert merged_data_ko.shape == original_shape
assert len(strategy_batches) == 90
assert len(omnibus_results) == len(PRIMARY_METRICS)
assert len(posthoc_results) == len(PRIMARY_METRICS) * 3
print('검정 계산 및 원본 보존 확인 완료')

전략별 평균


,최종농도(g/L),총수확량(kg),종료시간(h)
전략,,,
RC,24.3152,2.912750e+06,228.8333
OC,21.5084,2.833113e+06,229.2333
APC,29.4065,3.484283e+06,224.6667


Holm 보정 후 Welch ANOVA


,지표,Welch_F,Welch_p,Welch_p_Holm,Welch_판정
0,최종농도(g/L),16.112647,0.000005,0.000014,전략 간 차이 있음
1,총수확량(kg),15.965686,0.000005,0.000014,전략 간 차이 있음
2,종료시간(h),0.754552,0.474937,0.474937,차이 근거 부족


유의한 Games–Howell 비교


,지표,비교,평균차(A-B),95%CI_하한,95%CI_상한,GamesHowell_p,Hedges_g
1,최종농도(g/L),RC - APC,-5.091290,-9.167487,-1.015093,0.011613,-0.779347
2,최종농도(g/L),OC - APC,-7.898130,-11.682864,-4.113396,0.000033,-1.300546
4,총수확량(kg),RC - APC,-571533.333333,-939755.243964,-203311.422702,0.001505,-0.964591
5,총수확량(kg),OC - APC,-651170.000000,-982729.527142,-319610.472858,0.000069,-1.217710


검정 계산 및 원본 보존 확인 완료


최종 판단
현재 데이터로 내릴 수 있는 통계적 판단은 다음과 같습니다.
APC는 RC와 OC보다 최종 페니실린 농도가 높다.
APC는 RC와 OC보다 총수확량이 높다.
RC와 OC 사이에는 유의한 성과 차이가 확인되지 않았다.
전략별 종료시간 차이는 확인되지 않았다.
따라서 APC의 높은 성과가 단순히 운전시간 연장 때문이라고 볼 근거는 없다.
APC는 골든배치 기준 프로파일의 우선 후보가 될 수 있다.
다만 실제 운영 전략을 결정하기 전에는 다음을 추가로 판단해야 합니다.
총수확량 단위가 실제로 무엇인지
APC 배치가 다른 초기조건이나 공정조건을 가졌는지
APC의 높은 성과가 별도 Test 배치에서도 재현되는지
Fault 배치를 포함했을 때도 APC 공정 패턴이 안정적인지
최종농도·수확량 차이가 특정 소수 배치에 의해 만들어진 것은 아닌지
전체적으로 통계분석 방법과 계산 결과는 신뢰할 수 있지만, 전략 선택이나 인과적 결론에는 단위 확인과 외부 검증이 필요합니다

## 해석 시 주의

- p값은 차이의 크기가 아니므로 Hedges' g와 신뢰구간을 함께 본다.
- 이 결과는 관찰된 전략별 연관성이며 인과효과로 단정하지 않는다.
- Fault 배치는 전략 성과 비교에서 제외했으며 별도의 정상–Fault 분석 대상으로 남긴다.
- OUR 음수는 원본값을 유지했으며 이 노트북에서 클리핑하지 않았다.
- 어떤 CSV도 생성하거나 덮어쓰지 않는다.